<h1 style="font-weight: bold;">Python Flask Server Explanation</h1>

Create a subfolder **"artifacts"** in the server directory, and insert the pickle model, along with the class dictionary.  
Then create **server.py** and **util.py** file where we can write our backend flask server  

**The simple base for server.py is:**

In [ ]:
from flask import Flask, request, jsonify
import util

app = Flask(__name__)

@app.route('/hello')
def hello():
    return "hi"

if __name__ == "__main__":

    app.run(port=5000)

Now go to chrome and type **"http://localhost:5000/hello"**, you will find the text "hi" on that webpage after running **"server.py"**

Now, there are many ways to send image from frontend to backend for the processing.  
The method that we are using is, we are sending the images as **base64 encoded string**  
Base64 encoding strong basically represents the image in terms of text.  

***One of the ways to identify an image (without any gui) is mentioned below:-***

***server.py***

In [ ]:
#Importing the libraries
from flask import Flask, request, jsonify

#Creating the Flask app instance, enabling routing and handling requests.
app = Flask(__name__)

@app.route('/classify_image', methods=['GET','POST'])

#Function the classifies the images
def classify_image():
    return "hi"

if __name__ == "__main__":
    app.run(port=5000)

***util.py***

In [ ]:
import joblib
import json
import numpy as np
import base64
import cv2
from wavelet import w2d

__class_name_to_number = {}
__class_number_to_name = {}

__model = None

def classify_image(image_base64_data, file_path=None):

    imgs = get_cropped_image_if_2_eyes(file_path, image_base64_data)

    result = []
    for img in imgs:
        scalled_raw_img = cv2.resize(img, (32, 32))
        img_har = w2d(img, 'db1', 5)
        scalled_img_har = cv2.resize(img_har, (32, 32))
        combined_img = np.vstack((scalled_raw_img.reshape(32 * 32 * 3, 1), scalled_img_har.reshape(32 * 32, 1)))

        len_image_array = 32*32*3 + 32*32

        final = combined_img.reshape(1,len_image_array).astype(float)

        result.append(__model.predict(final)[0])

    return result
  
def load_saved_artifacts():
    print("loading saved artifacts...start")
    global __class_name_to_number
    global __class_number_to_name

    with open("server/artifacts/class_dictionary.json", "r") as f:
        __class_name_to_number = json.load(f)
        __class_number_to_name = {v:k for k,v in __class_name_to_number.items()}

    global __model
    if __model is None:
        with open("server/artifacts/saved_model.pkl", 'rb') as f:
            __model = joblib.load(f)
    print("loading saved artifacts...done")

def get_cv2_image_from_base64_string(b64str):
    encoded_data = b64str.split(',')[1]
    nparr = np.frombuffer(base64.b64decode(encoded_data), np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    return img

#This is a similar function used in our model
def get_cropped_image_if_2_eyes(image_path, image_base64_data):
    face_cascade = cv2.CascadeClassifier('server/opencv/haarcascades/haarcascade_frontalface_default.xml')
    eye_cascade = cv2.CascadeClassifier('server/opencv/haarcascades/haarcascade_eye.xml')

    if image_path:
        img = cv2.imread(image_path)
    else:
        img = get_cv2_image_from_base64_string(image_base64_data)

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    cropped_faces = []
    for (x,y,w,h) in faces:
            roi_gray = gray[y:y+h, x:x+w]
            roi_color = img[y:y+h, x:x+w]
            eyes = eye_cascade.detectMultiScale(roi_gray)
            if len(eyes) >= 2:
                cropped_faces.append(roi_color)
    return cropped_faces

#Function to return base64 strong of cristiano ronaldo's image
def get_b64_test_image_for_cristiano():
    with open("server/b64_cristiano.txt") as f:
        return f.read()
    
if __name__ == '__main__':
    load_saved_artifacts()
    print(classify_image(get_b64_test_image_for_cristiano(), None))

To test other athletes, just save their base64 encoded string within the same directory, copy their path in **get_b64_test_image_for_cristiano()** function, and run util.py.  
In the output it will return an array, that corresponds to the number in **class_dictionary.json**

{"cristiano_ronaldo": 0, "lionel_messi": 1, "neymar_jr": 2, "novak_djokovic": 3, "stephen_curry": 4}

<h5 style="font-weight: bold;">Printing probabilties of image classified</h5>

***util.py***

In [ ]:
import joblib
import json
import numpy as np
import base64
import cv2
from wavelet import w2d

__class_name_to_number = {}
__class_number_to_name = {}

__model = None

def classify_image(image_base64_data, file_path=None):

    imgs = get_cropped_image_if_2_eyes(file_path, image_base64_data)

    result = []
    for img in imgs:
        scalled_raw_img = cv2.resize(img, (32, 32))
        img_har = w2d(img, 'db1', 5)
        scalled_img_har = cv2.resize(img_har, (32, 32))
        combined_img = np.vstack((scalled_raw_img.reshape(32 * 32 * 3, 1), scalled_img_har.reshape(32 * 32, 1)))

        len_image_array = 32*32*3 + 32*32

        final = combined_img.reshape(1,len_image_array).astype(float)
        
        #These are the results that will be printed
        result.append({
            'class':class_number_to_name(__model.predict(final)[0]),
            'class_probabilty': __model.predict_proba(final)
            })

    return result
  
def load_saved_artifacts():
    print("loading saved artifacts...start")
    global __class_name_to_number
    global __class_number_to_name

    with open("server/artifacts/class_dictionary.json", "r") as f:
        __class_name_to_number = json.load(f)
        __class_number_to_name = {v:k for k,v in __class_name_to_number.items()}

    global __model
    if __model is None:
        with open("server/artifacts/saved_model.pkl", 'rb') as f:
            __model = joblib.load(f)
    print("loading saved artifacts...done")

#Converting the output number from 'class_dictionary.json' to a player name using this function.
def class_number_to_name(class_num):
    return __class_number_to_name[class_num]

def get_cv2_image_from_base64_string(b64str):
    encoded_data = b64str.split(',')[1]
    nparr = np.frombuffer(base64.b64decode(encoded_data), np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    return img

#This is a similar function used in our model
def get_cropped_image_if_2_eyes(image_path, image_base64_data):
    face_cascade = cv2.CascadeClassifier('server/opencv/haarcascades/haarcascade_frontalface_default.xml')
    eye_cascade = cv2.CascadeClassifier('server/opencv/haarcascades/haarcascade_eye.xml')

    if image_path:
        img = cv2.imread(image_path)
    else:
        img = get_cv2_image_from_base64_string(image_base64_data)

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    cropped_faces = []
    for (x,y,w,h) in faces:
            roi_gray = gray[y:y+h, x:x+w]
            roi_color = img[y:y+h, x:x+w]
            eyes = eye_cascade.detectMultiScale(roi_gray)
            if len(eyes) >= 2:
                cropped_faces.append(roi_color)
    return cropped_faces

#Function to return base64 strong of cristiano ronaldo's image
def get_b64_test_image_for_cristiano():
    with open("server/b64_cristiano.txt") as f:
        return f.read()
    
if __name__ == '__main__':
    load_saved_artifacts()
    print(classify_image(get_b64_test_image_for_cristiano(), None))